<a href="https://colab.research.google.com/github/Decoding-Data-Science/aiguild/blob/main/DDS_HR_Policy_Minimal_MCP_RAG_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DDS HR Policy Assistant — Minimal MCP + Dropbox + RAG Colab

This is the clean minimal version for your MCP demo.

**Required Colab Secrets**
- `DROPBOX_ACCESS_TOKEN`
- `OPENAI_API_KEY` or `OpenAI` or `OPENAI`

For Dropbox App Folder access, keep:

```python
DROPBOX_FOLDER_PATH = ""
```

## Cell 1 — Install packages

In [ ]:
%pip install -q \
  dropbox \
  fastmcp \
  langchain \
  langchain-openai \
  langchain-community \
  langchain-text-splitters \
  langchain-mcp-adapters \
  faiss-cpu \
  pypdf \
  gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.1/572.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 738.6/738.6 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.9/343.9 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.3/162.3 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 219.0/219.0 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.4/142.4 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4

## Cell 2 — Load secrets

In [ ]:
import os
from google.colab import userdata

OPENAI_API_KEY = (
    userdata.get("openai")
  )

DROPBOX_ACCESS_TOKEN = userdata.get("DROPBOX_ACCESS_TOKEN")

if not OPENAI_API_KEY:
    raise ValueError("OpenAI key not found. Use secret name OPENAI_API_KEY or OpenAI.")

if not DROPBOX_ACCESS_TOKEN:
    raise ValueError("Dropbox token not found. Use secret name DROPBOX_ACCESS_TOKEN.")

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

# For Dropbox App Folder apps, root path is empty string.
DROPBOX_FOLDER_PATH = ""

print("✅ Secrets loaded")
print("✅ Dropbox folder path:", repr(DROPBOX_FOLDER_PATH))

✅ Secrets loaded
✅ Dropbox folder path: ''


## Cell 3 — Create the Dropbox MCP server

In [ ]:
%%writefile /content/dds_dropbox_mcp_server.py

import os
import dropbox
from dropbox.files import FileMetadata, FolderMetadata
from fastmcp import FastMCP

mcp = FastMCP("DDS Dropbox HR Policy MCP Server")


def get_dropbox_client():
    token = os.getenv("DROPBOX_ACCESS_TOKEN")
    if not token:
        raise RuntimeError("DROPBOX_ACCESS_TOKEN is missing.")
    return dropbox.Dropbox(token)


@mcp.tool()
def dropbox_whoami() -> dict:
    """Return the connected Dropbox account."""
    dbx = get_dropbox_client()
    account = dbx.users_get_current_account()
    return {
        "email": account.email,
        "display_name": account.name.display_name,
    }


@mcp.tool()
def list_dropbox_entries(folder_path: str = "", recursive: bool = True) -> list:
    """List files/folders from the Dropbox App Folder."""
    dbx = get_dropbox_client()

    wrong_paths = {
        "/DDS_HR_Policy",
        "DDS_HR_Policy",
        "/Apps/DDS_HR_Policy",
        "Apps/DDS_HR_Policy",
        "/apps_ds_hr_policy",
        "apps_ds_hr_policy",
    }

    if folder_path in wrong_paths:
        folder_path = ""

    result = dbx.files_list_folder(folder_path, recursive=recursive)
    entries = list(result.entries)

    while result.has_more:
        result = dbx.files_list_folder_continue(result.cursor)
        entries.extend(result.entries)

    output = []

    for entry in entries:
        if isinstance(entry, FolderMetadata):
            output.append({
                "type": "folder",
                "name": entry.name,
                "path": entry.path_display,
                "size": None,
            })

        elif isinstance(entry, FileMetadata):
            output.append({
                "type": "file",
                "name": entry.name,
                "path": entry.path_display,
                "size": entry.size,
            })

    return output


@mcp.tool()
def download_dropbox_file_to_local(
    file_path: str,
    output_dir: str = "/content/dropbox_mcp_docs"
) -> dict:
    """Download one Dropbox file through MCP into Colab temporary storage."""
    dbx = get_dropbox_client()
    os.makedirs(output_dir, exist_ok=True)

    metadata, response = dbx.files_download(file_path)

    safe_name = file_path.strip("/").replace("/", "__")
    local_path = os.path.join(output_dir, safe_name)

    with open(local_path, "wb") as f:
        f.write(response.content)

    return {
        "name": metadata.name,
        "dropbox_path": metadata.path_display,
        "local_path": local_path,
        "size": metadata.size,
    }


if __name__ == "__main__":
    mcp.run(
        transport="streamable-http",
        host="127.0.0.1",
        port=8000,
        path="/mcp",
    )

Overwriting /content/dds_dropbox_mcp_server.py


## Cell 4 — Start the MCP HTTP server

In [ ]:
import os
import time
import subprocess

try:
    MCP_SERVER_PROCESS.terminate()
    MCP_SERVER_PROCESS.wait(timeout=3)
    print("Stopped old MCP server.")
except Exception:
    pass

server_env = os.environ.copy()
server_env["DROPBOX_ACCESS_TOKEN"] = DROPBOX_ACCESS_TOKEN

MCP_SERVER_PROCESS = subprocess.Popen(
    ["python", "/content/dds_dropbox_mcp_server.py"],
    env=server_env,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
)

time.sleep(5)

if MCP_SERVER_PROCESS.poll() is not None:
    stdout, stderr = MCP_SERVER_PROCESS.communicate(timeout=3)
    print("STDOUT:", stdout)
    print("STDERR:", stderr)
    raise RuntimeError("MCP server failed to start.")

print("✅ MCP server running at http://127.0.0.1:8000/mcp")

Stopped old MCP server.
✅ MCP server running at http://127.0.0.1:8000/mcp


## Cell 5 — Connect LangChain to MCP

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

try:
    client = MultiServerMCPClient(
        {
            "dds_dropbox": {
                "transport": "http",
                "url": "http://127.0.0.1:8000/mcp",
            }
        }
    )

    mcp_tools = await client.get_tools()

except Exception:
    client = MultiServerMCPClient(
        {
            "dds_dropbox": {
                "transport": "streamable_http",
                "url": "http://127.0.0.1:8000/mcp",
            }
        }
    )

    mcp_tools = await client.get_tools()

tools_by_name = {tool.name: tool for tool in mcp_tools}

print("✅ MCP tools loaded:")
for tool in mcp_tools:
    print("-", tool.name)

✅ MCP tools loaded:
- dropbox_whoami
- list_dropbox_entries
- download_dropbox_file_to_local


## Cell 6 — Normalize MCP output

In [ ]:
import json
import ast

def parse_text_payload(text):
    if not isinstance(text, str):
        return text

    text = text.strip()

    try:
        return json.loads(text)
    except Exception:
        pass

    try:
        return ast.literal_eval(text)
    except Exception:
        pass

    return text


def normalize_mcp_output(result):
    # Handles MCP wrapped output: [{'type': 'text', 'text': '[...]'}]
    if isinstance(result, list):
        if len(result) == 1:
            first = result[0]

            if isinstance(first, dict) and "text" in first:
                return parse_text_payload(first["text"])

            if hasattr(first, "text"):
                return parse_text_payload(first.text)

        return result

    if isinstance(result, dict):
        if "content" in result:
            return normalize_mcp_output(result["content"])

        if "text" in result:
            return parse_text_payload(result["text"])

        return result

    if hasattr(result, "text"):
        return parse_text_payload(result.text)

    if isinstance(result, str):
        return parse_text_payload(result)

    return result

## Cell 7 — List Dropbox files through MCP

In [ ]:
entries_raw = await tools_by_name["list_dropbox_entries"].ainvoke(
    {
        "folder_path": DROPBOX_FOLDER_PATH,
        "recursive": True,
    }
)

entries = normalize_mcp_output(entries_raw)

print(f"✅ Dropbox entries found: {len(entries)}")
print("-" * 80)

for entry in entries:
    print(entry["type"], "|", entry["name"], "|", entry["path"])

✅ Dropbox entries found: 4
--------------------------------------------------------------------------------
file | DDS_Leave_Policy_Synthetic_v1.pdf | /DDS_Leave_Policy_Synthetic_v1.pdf
file | DDS_Employee_Handbook_Synthetic_v1.pdf | /DDS_Employee_Handbook_Synthetic_v1.pdf
file | DDS_HR_FAQ_Synthetic_v1.pdf | /DDS_HR_FAQ_Synthetic_v1.pdf
file | DDS_Remote_Work_Policy_Synthetic_v1.pdf | /DDS_Remote_Work_Policy_Synthetic_v1.pdf


## Cell 8 — Select and download PDF files through MCP

In [ ]:
ALLOWED_EXTENSIONS = (".pdf", ".txt", ".md", ".docx", ".csv")

file_entries = [
    entry for entry in entries
    if entry.get("type") == "file"
    and entry.get("name", "").lower().endswith(ALLOWED_EXTENSIONS)
]

print(f"✅ Document files selected for RAG: {len(file_entries)}")
print("-" * 80)

for entry in file_entries:
    print(entry["name"], "|", entry["path"])


LOCAL_DOCS_DIR = "/content/dropbox_mcp_docs"
downloaded_files = []

for entry in file_entries:
    downloaded_raw = await tools_by_name["download_dropbox_file_to_local"].ainvoke(
        {
            "file_path": entry["path"],
            "output_dir": LOCAL_DOCS_DIR,
        }
    )

    downloaded = normalize_mcp_output(downloaded_raw)
    downloaded_files.append(downloaded)

    print("✅ Downloaded through MCP:", downloaded["local_path"])

print("\nTotal downloaded files:", len(downloaded_files))

✅ Document files selected for RAG: 4
--------------------------------------------------------------------------------
DDS_Leave_Policy_Synthetic_v1.pdf | /DDS_Leave_Policy_Synthetic_v1.pdf
DDS_Employee_Handbook_Synthetic_v1.pdf | /DDS_Employee_Handbook_Synthetic_v1.pdf
DDS_HR_FAQ_Synthetic_v1.pdf | /DDS_HR_FAQ_Synthetic_v1.pdf
DDS_Remote_Work_Policy_Synthetic_v1.pdf | /DDS_Remote_Work_Policy_Synthetic_v1.pdf
✅ Downloaded through MCP: /content/dropbox_mcp_docs/DDS_Leave_Policy_Synthetic_v1.pdf
✅ Downloaded through MCP: /content/dropbox_mcp_docs/DDS_Employee_Handbook_Synthetic_v1.pdf
✅ Downloaded through MCP: /content/dropbox_mcp_docs/DDS_HR_FAQ_Synthetic_v1.pdf
✅ Downloaded through MCP: /content/dropbox_mcp_docs/DDS_Remote_Work_Policy_Synthetic_v1.pdf

Total downloaded files: 4


## Cell 9 — Extract text from documents

In [ ]:
from pypdf import PdfReader

def extract_text_from_pdf(file_path):
    reader = PdfReader(file_path)
    pages = []

    for page_number, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""
        pages.append(f"\n--- Page {page_number} ---\n{text}")

    return "\n".join(pages)


raw_documents = []

for item in downloaded_files:
    local_path = item["local_path"]
    source_name = item["name"]
    dropbox_path = item["dropbox_path"]

    if not local_path.lower().endswith(".pdf"):
        print("Skipping non-PDF file:", source_name)
        continue

    text = extract_text_from_pdf(local_path)

    if text.strip():
        raw_documents.append(
            {
                "source": source_name,
                "dropbox_path": dropbox_path,
                "text": text,
            }
        )

        print("✅ Extracted:", source_name)
        print("Characters:", len(text))
        print("-" * 80)

    else:
        print("⚠️ No text extracted:", source_name)

print("\nTotal extracted documents:", len(raw_documents))

✅ Extracted: DDS_Leave_Policy_Synthetic_v1.pdf
Characters: 3643
--------------------------------------------------------------------------------
✅ Extracted: DDS_Employee_Handbook_Synthetic_v1.pdf
Characters: 8009
--------------------------------------------------------------------------------
✅ Extracted: DDS_HR_FAQ_Synthetic_v1.pdf
Characters: 1575
--------------------------------------------------------------------------------
✅ Extracted: DDS_Remote_Work_Policy_Synthetic_v1.pdf
Characters: 3054
--------------------------------------------------------------------------------

Total extracted documents: 4


## Cell 10 — Chunk documents

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
)

docs = []

for doc in raw_documents:
    chunks = splitter.split_text(doc["text"])

    for i, chunk in enumerate(chunks):
        docs.append(
            Document(
                page_content=chunk,
                metadata={
                    "source": doc["source"],
                    "dropbox_path": doc["dropbox_path"],
                    "chunk": i,
                },
            )
        )

print("✅ Total chunks:", len(docs))

✅ Total chunks: 27


## Cell 11 — Create vector database

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

if not docs:
    raise ValueError("No chunks found. Check whether PDF extraction worked.")

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = FAISS.from_documents(
    documents=docs,
    embedding=embeddings,
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

print("✅ Vector database created")

/tmp/ipykernel_9475/1293596575.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


✅ Vector database created


## Cell 12 — Build RAG answer function

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)


def answer_hr_question(question):
    relevant_docs = retriever.invoke(question)

    context = "\n\n".join(
        [
            f"Source: {doc.metadata['source']}\nContent:\n{doc.page_content}"
            for doc in relevant_docs
        ]
    )

    prompt = f"""
You are the DDS HR Policy Assistant.

Answer the question using only the HR policy context below.

If the answer is not available in the context, say:
"I could not find this information in the available HR policy documents."

Mention the source document names used.

Context:
{context}

Question:
{question}

Answer:
"""

    response = llm.invoke(prompt)
    return response.content

## Cell 13 — Test

In [ ]:
answer_hr_question("What is the standard working hours in Decoding data science?")

'The standard office hours at Decoding Data Science (DDS) are 9:00–18:00, Monday–Friday. \n\nSource document: DDS_Employee_Handbook_Synthetic_v1.pdf'

## Cell 14 — Launch Gradio chatbot

In [ ]:
import gradio as gr

def chat_fn(message, history):
    return answer_hr_question(message)

demo = gr.ChatInterface(
    fn=chat_fn,
    title="DDS HR Policy Assistant — MCP + Dropbox + RAG",
    description="Dropbox files are accessed through MCP, then indexed with RAG.",
)

demo.launch(debug=True, share=True)

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://4332a77a8b035566d6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
